# 🧭 Routing with an LLM Classifier

**Routing** picks *where a query should go* before answering it. A single general-purpose prompt
gives mediocre answers across many domains; a router sends each question to a **specialist prompt**
tuned for that domain.

This notebook routes between four experts — personal finance, book reviews, health & fitness, and
travel — by asking an LLM to classify the question first.

## Learning Objectives
1. **Why route at all** — specialist prompts beat one generic prompt across mixed domains
2. **LLM-as-classifier** — using a model's reasoning to choose a destination
3. **Why string matching breaks** — see the original implementation fail, then fix it
4. **Structured output routing** — `Literal` types make invalid routes unrepresentable
5. **Inspecting decisions** — surfacing the router's reasoning, especially on ambiguous queries

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Familiarity with LCEL chains (`prompt | llm | parser`)
- Sibling notebooks: `b. Semantic_Routing` and `c. Self_Querying_Retrieval`

---
## 🧠 Part 1: What Routing Solves

One prompt cannot be excellent at everything. A prompt that opens *"You are a certified health and
fitness expert"* produces better nutrition advice than a generic assistant — but it is the wrong
persona entirely for a question about index funds.

Routing splits the problem in two:

| Stage | Job | Cost |
|---|---|---|
| **Route** | Decide which specialist should answer | One classification call |
| **Answer** | Run the specialist's prompt | One generation call |

### Key Concepts:
- **Destination**: one of several specialist prompts the router can select.
- **Classifier**: the component that maps a query to a destination.
- **Fallback**: what happens when the classifier is unsure or returns something unexpected.

> **Key Insight**: routing costs an extra LLM call on *every* query. It pays off when the
> destinations are genuinely different in behavior — not merely different in wording.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# LangChain core
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

Credentials come from the repo-root `.env` via `load_dotenv()` — never hard-code a key in a cell.

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK checks the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already present in `.env` would silently
> win and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Routing-LLM-Classifier"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the LLM

The same model does both jobs here — classification and generation. In production these are often
split: a small, cheap model routes, and a larger one answers.

In [ ]:
# ============================================================================
# MODEL INITIALIZATION
# ============================================================================
llm = get_experientiallabs_llm()

print(f"🤖 LLM: {llm.model_name}")

---
## 📝 Part 3: The Destination Prompts

Four specialist prompts, each establishing a different persona and expertise. These are the
router's possible destinations. Every one takes the same `{query}` variable, which is what makes
them interchangeable at the end of the chain.

In [ ]:
# ============================================================================
# DESTINATIONS: One specialist prompt per domain
# ============================================================================
personal_finance_template = """You are a personal finance expert with extensive knowledge of budgeting, investing, and financial planning. You offer clear and practical advice on managing money and making sound financial decisions.

Here is a question:
{query}"""

book_review_template = """You are an experienced book critic with extensive knowledge of literature, genres, and authors. You provide thoughtful and analytical reviews and insights about books.

Here is a question:
{query}"""

health_fitness_template = """You are a certified health and fitness expert with a deep understanding of nutrition, exercise routines, and wellness strategies. You offer practical and evidence-based advice about health and fitness.

Here is a question:
{query}"""

travel_guide_template = """You are a seasoned travel expert with extensive knowledge of destinations, travel tips, and cultural insights. You provide detailed and useful advice about travel.

Here is a question:
{query}"""

# Keyed by the route name the classifier will emit — this mapping IS the router table.
PROMPTS = {
    "personal_finance": personal_finance_template,
    "book_review": book_review_template,
    "health_fitness": health_fitness_template,
    "travel_guide": travel_guide_template,
}

print(f"✅ {len(PROMPTS)} destinations defined: {', '.join(PROMPTS)}")

---
## ⚠️ Part 4: The Fragile Approach (and Why It Fails)

The original version of this notebook classified with a plain string prompt and then compared the
result using **exact equality**:

```python
classification = classification_chain.invoke({"question": query})

if classification == "Personal Finance":
    ...
elif classification == "Book Review":
    ...
```

Run the cell below and look at the raw output before reading on.

In [ ]:
# ============================================================================
# FRAGILE CLASSIFIER: String output compared with exact equality
# ============================================================================
classification_template = PromptTemplate.from_template(
    """You are good at classifying a question.
    Given the user question below, classify it as either being about personal finance, book reviews, health & fitness, or travel guides.

    <question>
    {question}
    </question>

    Classification:"""
)

classification_chain = classification_template | llm | StrOutputParser()

for q in [
    "What are effective strategies for losing weight?",
    "What are the must-see attractions in USA?",
]:
    raw = classification_chain.invoke({"question": q})
    print(f"❓ {q}")
    print(f"   raw output          : {raw!r}")
    print(f"   == 'Health & Fitness'? {raw == 'Health & Fitness'}")
    print(f"   == 'Travel Guide'?     {raw == 'Travel Guide'}\n")

### What Went Wrong

The model returns `'health & fitness'` (lowercase) and `'travel guides'` (plural). Neither matches
the exact literals the original code tested for, so **every branch falls through to `else`** and the
router returns `None`. The notebook appeared to work only because the failure was silent.

Nothing here is the model misbehaving — it answered the question correctly. The bug is that the
code demanded an exact surface form that the prompt never guaranteed.

| Failure | Cause |
|---|---|
| `'health & fitness'` ≠ `'Health & Fitness'` | Capitalization drift |
| `'travel guides'` ≠ `'Travel Guide'` | Pluralization drift |
| Trailing whitespace / `"Classification: X"` | Formatting drift |

> **Rule of thumb**: the moment you compare an LLM's free text against a fixed literal, you have a
> latent bug. Constrain the output instead of hoping it matches.

---
## ✅ Part 5: The Robust Approach — Structured Output

`with_structured_output()` binds a Pydantic schema to the model. Typing `destination` as a
`Literal` makes any value outside those four **unrepresentable** — the provider is constrained to
one of them, and LangChain validates before you ever see it.

The `reasoning` field costs almost nothing and turns the router from a black box into something you
can debug.

In [ ]:
# ============================================================================
# ROBUST CLASSIFIER: Literal-constrained structured output
# ============================================================================
class RouteDecision(BaseModel):
    """Which specialist should handle this question."""

    destination: Literal[
        "personal_finance", "book_review", "health_fitness", "travel_guide"
    ] = Field(description="The expert best suited to answer the question")
    reasoning: str = Field(description="One short sentence explaining the choice")


router_prompt = PromptTemplate.from_template(
    """Classify the user's question and choose the expert best suited to answer it.

    - personal_finance : budgeting, investing, saving, financial planning
    - book_review      : literature, genres, authors, literary analysis
    - health_fitness   : nutrition, exercise, wellness strategies
    - travel_guide     : destinations, travel tips, cultural insights

    Question: {question}"""
)

route_chain = router_prompt | llm.with_structured_output(RouteDecision)

decision = route_chain.invoke({"question": "What are effective strategies for losing weight?"})

print(f"✅ destination : {decision.destination}")
print(f"   reasoning   : {decision.reasoning}")
print(f"   type        : {type(decision).__name__} (validated, not raw text)")

---
## 🔬 Part 6: Look Inside the Router

A router is only trustworthy if you can see its decisions. The queries below are deliberately
varied: three unambiguous, then two that straddle a boundary.

Watch the last two. *"Is the food in Tokyo worth budgeting a whole trip around?"* touches travel,
finance, **and** food — there is no objectively correct answer, and the `reasoning` field is what
lets you judge whether the choice was defensible.

In [ ]:
# ============================================================================
# INTROSPECTION: Routing decisions with the model's reasoning
# ============================================================================
probes = [
    "How do I start investing in the stock market?",
    "What makes a novel a classic?",
    "How often should I exercise to maintain good health?",
    # --- deliberately ambiguous ---
    "Is the food in Tokyo worth budgeting a whole trip around?",
    "How much should I set aside each month for a gym membership?",
]

for q in probes:
    d = route_chain.invoke({"question": q})
    print(f"❓ {q}")
    print(f"   🧭 {d.destination}")
    print(f"   💭 {d.reasoning}\n")

---
## 🔗 Part 7: The Full Routed Chain

`RunnableLambda` lets an ordinary Python function sit inside an LCEL chain. Here it classifies the
query and returns the **selected prompt**, which the rest of the chain then executes.

Because `destination` is a validated `Literal`, the dictionary lookup cannot raise a `KeyError` —
the `else`/fallback branch that the original code always hit is now unreachable by construction.

In [ ]:
# ============================================================================
# ROUTED CHAIN: classify -> select prompt -> answer
# ============================================================================
def prompt_router(input_dict):
    """Classify the query and return the matching specialist prompt."""
    decision = route_chain.invoke({"question": input_dict["query"]})
    print(f"🧭 Routed to: {decision.destination}  ({decision.reasoning})")
    return PromptTemplate.from_template(PROMPTS[decision.destination])


full_chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

print("✅ Routed chain ready")

### 7.1 Run It Across All Four Domains

One loop instead of four near-identical cells. Each answer should carry the voice of its
specialist prompt — compare the tone of the finance answer against the travel one.

In [ ]:
# ============================================================================
# END TO END: One query per destination
# ============================================================================
queries = [
    "What are the must-see attractions in the USA?",
    "What makes a novel a classic?",
    "What are effective strategies for losing weight?",
    "What are the best strategies for saving money?",
]

for q in queries:
    print(f"\n{'=' * 78}\n❓ {q}\n{'=' * 78}")
    print(full_chain.invoke(q)[:600], "...")

---
## 🚧 Part 8: Failure Modes Worth Knowing

Structured output removes *format* errors. It does not remove *judgment* errors.

| Failure | What happens | Mitigation |
|---|---|---|
| **Forced choice** | A `Literal` guarantees one of the four is returned — even for *"What time is it?"* | Add an explicit `"other"` / `"none"` destination |
| **Ambiguous queries** | Model picks one plausible route; the other was equally valid | Log `reasoning`; consider multi-route fan-out |
| **Route drift** | Adding a destination changes behavior on existing queries | Keep a regression set of query → expected route |
| **Latency and cost** | Every query pays a classification call before answering | Route with a smaller/cheaper model than you answer with |

> **Note**: the forced-choice problem is the one most often overlooked. A router with no escape
> hatch will confidently misroute anything outside its taxonomy.

In [ ]:
# ============================================================================
# FAILURE DEMO: An out-of-scope question still gets a route
# ============================================================================
off_topic = "What is the capital of France?"
d = route_chain.invoke({"question": off_topic})

print(f"❓ {off_topic}")
print(f"   🧭 {d.destination}")
print(f"   💭 {d.reasoning}")
print("\n⚠️  The Literal type guarantees a valid route — not a sensible one.")
print("   A production router needs an explicit 'other' destination as an escape hatch.")

---
## 📝 Summary

### 1. Why Route
- Specialist prompts outperform one generic prompt across mixed domains. Routing buys that at the
  cost of one extra LLM call per query.

### 2. The Bug in the Original Implementation
- Comparing free-text output against exact literals (`classification == "Health & Fitness"`) failed
  on every query: the model returned `'health & fitness'` and `'travel guides'`.
- The failure was **silent** — every branch fell through to `else` and the router returned `None`.

### 3. The Fix: Constrain, Don't Hope
- `with_structured_output()` + a `Literal` type makes invalid destinations unrepresentable.
- Adding a `reasoning` field costs almost nothing and makes the router debuggable.

### 4. Inspect Ambiguous Cases
- Unambiguous queries tell you nothing. Boundary-straddling queries — travel vs. finance vs. food —
  are where you learn whether your taxonomy actually holds.

### 5. Remaining Failure Modes
- A `Literal` guarantees a **valid** route, not a **sensible** one. Out-of-scope questions are
  still confidently routed; add an explicit `"other"` destination.

### 6. How This Compares to Its Siblings
| Notebook | Routes on | Decides by | Cost per query |
|---|---|---|---|
| **`a. Routing_LLM_Classifier`** (this one) | Which **prompt/data source** | LLM reasoning | One extra LLM call |
| **`b. Semantic_Routing`** | Which **prompt** | Embedding distance to example queries | One embedding call (~100× cheaper) |
| **`c. Self_Querying_Retrieval`** | Nothing — one source | Builds a **metadata filter** instead | One LLM call |

- **LLM routing** handles nuance, novel phrasings, and multi-clause questions — but costs a
  generation call and can hallucinate a plausible-but-wrong route.
- **Semantic routing** is fast and cheap, but cannot *reason*: it only measures similarity to
  example queries, so a question phrased unlike any example routes badly.
- **Self-querying** is not really routing at all — it narrows *within* one source rather than
  choosing between sources.

### Next Steps
- Inspect these runs in LangSmith under the **Routing-LLM-Classifier** project; each query shows
  the classification call followed by the generation call.
- Read `b. Semantic_Routing` next and compare the two decisions on the same query.